In [1]:
# Get descriptive statistics.

import zipfile
from PIL import Image
import io
import numpy as np

zip_path = "deepfake-vs-real-60k.zip"

res_real = []
res_fake = []

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    files = [f for f in z.namelist() if f.lower().endswith((".png", ".jpg"))]

    for fname in files:
        count += 1

        # Print progress every 10000 files
        if count % 10000 == 0:
            print(f"Processed {count} images...")

        with z.open(fname) as f:
            img = Image.open(io.BytesIO(f.read()))
            w, h = img.size

            if fname.lower().endswith(".png"):
                res_real.append((w, h))
            else:
                res_fake.append((w, h))

print("Done!")

# Convert to arrays for stats
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL (PNG) ===")
print("Count:", len(real_arr))
print("Min:", real_arr.min(axis=0))
print("Max:", real_arr.max(axis=0))
print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE (JPG) ===")
print("Count:", len(fake_arr))
print("Min:", fake_arr.min(axis=0))
print("Max:", fake_arr.max(axis=0))
print("Mean:", fake_arr.mean(axis=0))

Processed 10000 images...
Processed 20000 images...
Processed 30000 images...
Processed 40000 images...
Processed 50000 images...
Done!
=== REAL (PNG) ===
Count: 28475
Min: [512 512]
Max: [512 512]
Mean: [512. 512.]

=== FAKE (JPG) ===
Count: 28596
Min: [512 512]
Max: [4284 5712]
Mean: [1155.21006434 1582.98622185]


In [2]:
# Unify file types and image resolutions.

import zipfile
from PIL import Image
import io
import os

zip_path = "deepfake-vs-real-60k.zip"
output_dir = "dataset_1_cleaned"
target_size = (256, 256)

os.makedirs(output_dir, exist_ok=True)

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    files = [f for f in z.namelist() if f.lower().endswith((".png", ".jpg"))]

    for fname in files:
        count += 1
        if count % 1000 == 0:
            print(f"Processed {count} images...")

        is_real = fname.lower().endswith(".png")
        cls = "real" if is_real else "fake"

        class_dir = os.path.join(output_dir, cls)
        os.makedirs(class_dir, exist_ok=True)

        with z.open(fname) as f:
            img = Image.open(io.BytesIO(f.read())).convert("RGB")

        img = img.resize(target_size, Image.LANCZOS)

        base = os.path.splitext(os.path.basename(fname))[0]

        # Add a suffix so names are unique and traceable
        suffix = "_real" if is_real else "_fake"
        out_name = base + suffix + ".jpg"
        out_path = os.path.join(class_dir, out_name)

        img.save(out_path, "JPEG", quality=95, subsampling=2)

print("Done! All images converted and resized.")

Processed 1000 images...
Processed 2000 images...
Processed 3000 images...
Processed 4000 images...
Processed 5000 images...
Processed 6000 images...
Processed 7000 images...
Processed 8000 images...
Processed 9000 images...
Processed 10000 images...
Processed 11000 images...
Processed 12000 images...
Processed 13000 images...
Processed 14000 images...
Processed 15000 images...
Processed 16000 images...
Processed 17000 images...
Processed 18000 images...
Processed 19000 images...
Processed 20000 images...
Processed 21000 images...
Processed 22000 images...
Processed 23000 images...
Processed 24000 images...
Processed 25000 images...
Processed 26000 images...
Processed 27000 images...
Processed 28000 images...
Processed 29000 images...
Processed 30000 images...
Processed 31000 images...
Processed 32000 images...
Processed 33000 images...
Processed 34000 images...
Processed 35000 images...
Processed 36000 images...
Processed 37000 images...
Processed 38000 images...
Processed 39000 image

In [3]:
# Validate updated dataset.

import os
from PIL import Image
import numpy as np

cleaned_path = "dataset_1_cleaned"

real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

res_real = []
res_fake = []

# --- REAL IMAGES ---
real_files = [f for f in os.listdir(real_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(real_files):
    if i % 10000 == 0:
        print(f"Processed {i} real images...")

    img_path = os.path.join(real_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_real.append((w, h))

# --- FAKE IMAGES ---
fake_files = [f for f in os.listdir(fake_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(fake_files):
    if i % 10000 == 0:
        print(f"Processed {i} fake images...")

    img_path = os.path.join(fake_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_fake.append((w, h))

print("Done!")

# Convert to arrays for stats
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL (cleaned JPG) ===")
print("Count:", len(real_arr))
print("Min:", real_arr.min(axis=0))
print("Max:", real_arr.max(axis=0))
print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE (cleaned JPG) ===")
print("Count:", len(fake_arr))
print("Min:", fake_arr.min(axis=0))
print("Max:", fake_arr.max(axis=0))
print("Mean:", fake_arr.mean(axis=0))

Processed 0 real images...
Processed 10000 real images...
Processed 20000 real images...
Processed 0 fake images...
Processed 10000 fake images...
Processed 20000 fake images...
Done!
=== REAL (cleaned JPG) ===
Count: 28475
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]

=== FAKE (cleaned JPG) ===
Count: 28596
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]


In [4]:
# Color comparison between real and fake images.

import os
import numpy as np
from PIL import Image

cleaned_path = "dataset_1_cleaned"
real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

def compute_color_stats(folder):
    means = []
    stds = []

    files = [f for f in os.listdir(folder) if f.lower().endswith(".jpg")]

    for i, fname in enumerate(files):
        if i % 10000 == 0:
            print(f"Processed {i} images in {folder}...")

        img = Image.open(os.path.join(folder, fname)).convert("RGB")
        arr = np.array(img) / 255.0  # normalize to [0,1]

        means.append(arr.mean(axis=(0,1)))
        stds.append(arr.std(axis=(0,1)))

    means = np.array(means)
    stds = np.array(stds)

    return means.mean(axis=0), stds.mean(axis=0)

real_mean, real_std = compute_color_stats(real_dir)
fake_mean, fake_std = compute_color_stats(fake_dir)

print("\n=== REAL ===")
print("Mean RGB:", real_mean)
print("Std RGB:", real_std)

print("\n=== FAKE ===")
print("Mean RGB:", fake_mean)
print("Std RGB:", fake_std)

Processed 0 images in dataset_1_cleaned\real...
Processed 10000 images in dataset_1_cleaned\real...
Processed 20000 images in dataset_1_cleaned\real...
Processed 0 images in dataset_1_cleaned\fake...
Processed 10000 images in dataset_1_cleaned\fake...
Processed 20000 images in dataset_1_cleaned\fake...

=== REAL ===
Mean RGB: [0.52026528 0.42533899 0.38030437]
Std RGB: [0.25246236 0.22657993 0.22372848]

=== FAKE ===
Mean RGB: [0.47030001 0.40848903 0.38178443]
Std RGB: [0.23700872 0.21752416 0.20646516]
